# Bolo 01: synthetic Hinglish command speech (Indic Parler-TTS)

Generates 16 kHz training/dev/test speech for the Bolo command dataset with AI4Bharat **Indic Parler-TTS**
(Apache-2.0, gated). Voices are split by speaker: test voices (Aman, Rani, Priya, Vikas) never appear in train.

**Before running**
1. On huggingface.co, open `ai4bharat/indic-parler-tts` and accept the access conditions.
2. Create a read token and add it as a notebook secret named `HF_TOKEN`.
3. Kaggle: *Add input* the private dataset **bolo-bundle** (made by `scripts/make_kaggle_bundle.py`).
   Settings: **GPU T4 x2** (one is used), **Internet on**.
4. Use **Save Version -> Save & Run All (Commit)** so it runs in the background (it is resumable: if it
   stops, re-run and it continues from `manifest.jsonl`).

Expected time: unknown until measured; the progress log prints seconds/utterance and an ETA. Lower `N`
below to try a small run first (e.g. 64 per split, a few minutes).
Output: `/kaggle/working/bolo-tts/` (audio + `manifest.jsonl` + `rejected.jsonl`), roughly 0.5 GB.

In [ ]:
!pip install -q git+https://github.com/huggingface/parler-tts.git soundfile

In [ ]:
import glob, os, sys, json
# Kaggle: bundle uploaded as a private dataset -> /kaggle/input/...; Colab: unzip bolo-bundle.zip into /content
cands = glob.glob("/kaggle/input/**/bolo-bundle/src/bolo/__init__.py", recursive=True) + \
        glob.glob("/content/**/bolo-bundle/src/bolo/__init__.py", recursive=True)
assert cands, "bolo-bundle not found: add the bolo-bundle dataset (Kaggle) or unzip it in /content (Colab)"
BUNDLE = cands[0].split("/src/bolo/")[0]
sys.path.insert(0, BUNDLE + "/src")
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
print("bundle:", BUNDLE, "| work dir:", WORK)

In [ ]:
# Indic Parler-TTS is gated: accept its terms on huggingface.co first, then add a READ token as a secret
# named HF_TOKEN (Kaggle: Add-ons -> Secrets; Colab: key icon -> Secrets).
token, errs = None, []
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    errs.append(f"kaggle: {type(e).__name__}: {e}")
if not token and not os.path.exists("/kaggle"):
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception as e:
        errs.append(f"colab: {type(e).__name__}: {e}")
token = token or os.environ.get("HF_TOKEN")
assert token, "HF_TOKEN secret missing: " + " | ".join(errs)
from huggingface_hub import login
login(token=token)

In [ ]:
from bolo.audio import tts
N = {"train": 5000, "dev": 300, "test": 1000}   # max utterances per split; lower for a trial run
BATCH = 16                                     # lower to 8 if CUDA OOM
OUT = WORK + "/bolo-tts"
jobs = []
for split, n in N.items():
    rows = [json.loads(l) for l in open(f"{BUNDLE}/data/text/v0/{split}.jsonl", encoding="utf-8")][:n]
    jobs += tts.plan(rows, split, seed=0)
print(len(jobs), "jobs; example:", json.dumps(jobs[0], ensure_ascii=False)[:400])

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# 1) self-check on a small slice first: stop early (and save GPU quota) if the TTS output looks broken
probe = [j for j in jobs if j["split"] == "train"][:48] + [j for j in jobs if j["split"] == "test"][:16]
st = tts.synthesize(probe, OUT, batch_size=BATCH, device=DEVICE)
man = [json.loads(l) for l in open(OUT + "/manifest.jsonl")] if os.path.exists(OUT + "/manifest.jsonl") else []
rej = [json.loads(l) for l in open(OUT + "/rejected.jsonl")] if os.path.exists(OUT + "/rejected.jsonl") else []
rate = len(rej) / max(1, len(man) + len(rej))
print(f"probe: kept {len(man)}, rejected {len(rej)} ({rate:.0%})", [r["reject"] for r in rej][:10])
assert rate < 0.3, "too many TTS rejections in the probe: inspect rejected.jsonl before a full run"
# 2) full run (resumable; the probe utterances are skipped)
stats = tts.synthesize(jobs, OUT, batch_size=BATCH, device=DEVICE)
print(stats)

In [ ]:
import collections
man = [json.loads(l) for l in open(OUT + "/manifest.jsonl")]
rej = [json.loads(l) for l in open(OUT + "/rejected.jsonl")] if os.path.exists(OUT + "/rejected.jsonl") else []
print("kept per split:", collections.Counter(r["split"] for r in man))
print("rejected:", collections.Counter(r["reject"] for r in rej))
print("hours:", round(sum(r["duration_s"] for r in man) / 3600, 2))
json.dump({"kept": collections.Counter(r["split"] for r in man), "rejected": collections.Counter(r["reject"] for r in rej),
           "hours": sum(r["duration_s"] for r in man) / 3600}, open(OUT + "/summary.json", "w"), indent=1)

Listen to a few before training on them (TTS can mispronounce English words in Devanagari context):

In [ ]:
from IPython.display import Audio, display
import random
for r in random.Random(0).sample(man, 6):
    print(r["speaker"], "|", r["tts_input"], "| target:", r["asr_text"])
    display(Audio(OUT + "/" + r["audio"]))